# Chapter 16 &mdash; NP-Hard Can Be Undecidable: the Diophantine Pitfall

**Concept 10 of the Chapter 16 decomposition:** *NP-Hard Can Be Undecidable: the Diophantine Pitfall*

Diophantine equations are NP-hard <i>and</i> undecidable &mdash; so NP-hardness alone establishes nothing about membership in NP.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-NP-Hard-Can-Be-Undecidable/Concept-NP-Hard-Can-Be-Undecidable.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A trap worth naming. **NP-hard** says "at least as hard as everything in NP". It says
**nothing** about being *in* NP &mdash; or even about being decidable.

The witness is **Hilbert's tenth problem**: does a given polynomial equation with
integer coefficients have an integer solution? It is

* **NP-hard** &mdash; SAT reduces to it;
* **undecidable** &mdash; Matiyasevich, 1970.

So "NP-hard" is not a synonym for "hard but doable". The correct reading:

* **NP-hard** &mdash; a lower bound, nothing more;
* **NP-complete** &mdash; NP-hard **and in NP**, so decidable, and with short certificates.

Always check membership in NP before saying "NP-complete". Concept 7's step 1 exists
for exactly this reason.

## 2. Definitions

### A Diophantine solver, necessarily bounded

In [ ]:
from itertools import product
def diophantine_search(poly, nvars_, bound=12):
    # poly is a function of a tuple of integers
    for vals in product(range(-bound, bound + 1), repeat=nvars_):
        if poly(vals) == 0:
            return vals
    return None

### The classes, as a table

In [ ]:
CLAIMS = [
 ("in NP",         "short certificate, polynomial verifier"),
 ("NP-hard",       "every NP problem reduces to it -- a LOWER bound only"),
 ("NP-complete",   "both of the above"),
 ("undecidable",   "no decider exists at all"),
]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;9.&nbsp;Clique is NP-Complete: Reduction from 3-SAT](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Clique-Is-NPC/Concept-Clique-Is-NPC.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16-NPC/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;11.&nbsp;Co-NP and Co-NPC: Primes versus Composites](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Co-NP-And-Primes/Concept-Co-NP-And-Primes.ipynb)&nbsp;&rarr;

---

## 3. Tests

Some equations have solutions, and finding one is a search.

In [ ]:
eqs = [("x^2 - 4",            lambda v: v[0]**2 - 4,                      1),
       ("x^2 + y^2 - 25",     lambda v: v[0]**2 + v[1]**2 - 25,           2),
       ("x^3 + y^3 - z^3",    lambda v: v[0]**3 + v[1]**3 - v[2]**3,      3)]
for name, f, k in eqs:
    sol = diophantine_search(f, k, bound=6)
    print("  %-18s solution %s" % (name, sol))
assert diophantine_search(eqs[0][1], 1, bound=6) is not None

Some have none, and the search can only ever say 'not found yet'.

In [ ]:
none_eq = ("x^2 + y^2 + 3", lambda v: v[0]**2 + v[1]**2 + 3, 2)
for bound in [3, 8, 20]:
    print("  searched |x|,|y| <= %2d : %s"
          % (bound, diophantine_search(none_eq[1], 2, bound=bound)))
assert diophantine_search(none_eq[1], 2, bound=20) is None
print("\nHere we can PROVE there is none (a sum of squares is non-negative).")
print("In general, no such proof method exists -- that is Matiyasevich's theorem.")

**Unbounded search** is the problem: there is no bound to search up to.

In [ ]:
hard = lambda v: v[0]**2 - 991 * v[1]**2 - 1        # Pell's equation
for bound in [10, 60, 200]:
    print("  bound %3d : %s" % (bound, diophantine_search(hard, 2, bound=bound)))
print("\nx^2 - 991y^2 = 1 HAS a solution -- with x about 3.8 x 10^29.")
print("No bound you choose is big enough in general.")

So the four claims are genuinely different.

In [ ]:
for a, b in CLAIMS: print("  %-14s %s" % (a, b))
print()
print("%-26s %-10s %-10s %s" % ("problem", "in NP", "NP-hard", "decidable"))
for p, inp, hard_, dec in [("3-SAT", "yes", "yes", "yes"),
                           ("clique", "yes", "yes", "yes"),
                           ("Diophantine", "NO", "yes", "NO"),
                           ("halting", "NO", "yes", "NO"),
                           ("2-SAT", "yes", "no", "yes")]:
    print("%-26s %-10s %-10s %s" % (p, inp, hard_, dec))

Why Diophantine is not in NP.

In [ ]:
print("A certificate would be the integer solution.")
print("But the smallest solution can be ASTRONOMICALLY larger than the")
print("equation -- 10^29 for a three-digit coefficient -- so the certificate")
print("is not polynomially bounded in the input size.")
print()
print("No polynomial bound on the certificate => not in NP.")

The discipline this imposes.

In [ ]:
print("Before writing 'NP-complete', answer:")
print("  1. what is the certificate?")
print("  2. is it polynomially bounded in |x|?")
print("  3. can it be checked in polynomial time?")
print()
print("If you cannot answer all three, you have NP-hardness, not completeness.")

## 4. Exercises


1. Give a problem that is NP-hard, decidable, and not in NP.
2. Why is the halting problem NP-hard?
3. What is the certificate for Pell's equation, and how long is it?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16-NPC/Concept-NP-Hard-Can-Be-Undecidable')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')